<a href="https://colab.research.google.com/github/code-with-Akshaya/Smart-Market-Intelligence/blob/main/FlipKart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Competitive Gap Analysis for E‑Commerce Platforms**

## The challenge is to design an AI‑driven workflow that compares products across competing platforms (Flipkart vs Amazon), identifies pricing differences, highlights recurring customer complaints, and generates structured, goal‑aware reports with citations. This system must handle large review datasets, ensure accurate sentiment analysis even for long texts, and provide interactive filtering so decision‑makers can dynamically explore insights tailored to growth, margins, or retention objectives

##1.Data source

### -> Mount the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### -> Loading the data

In [ ]:
import pandas as pd

flipkart_df = pd.read_csv("/content/drive/MyDrive/Dataset/flipkart_com-ecommerce_sample.csv")
amazon_df = pd.read_csv("/content/drive/MyDrive/Dataset/amazon.csv")

print("Flipkart Dataset:\n")
print(flipkart_df.head())
print("\n-----------------------------------------------------------------------------------------------------")
print("\nAmazon Dataset:\n")
print(amazon_df.head())


In [ ]:
print("Flipkart:\n")
print(flipkart_df.columns)
print("\nAmazon:\n")
print(amazon_df.columns)


## 2. Pre-Processing: Cleaning the dataset

In [ ]:
# Clean price columns
flipkart_df['retail_price'] = pd.to_numeric(flipkart_df['retail_price'], errors='coerce')
flipkart_df['discounted_price'] = pd.to_numeric(flipkart_df['discounted_price'], errors='coerce')

# Keep useful columns
flipkart_df = flipkart_df[['product_name','brand','product_category_tree','retail_price','discounted_price','description']]


In [ ]:
# Clean price columns
amazon_df['actual_price'] = amazon_df['actual_price'].str.replace('₹','').str.replace(',','').astype(float)
amazon_df['discounted_price'] = amazon_df['discounted_price'].str.replace('₹','').str.replace(',','').astype(float)

# Clean reviews
amazon_df['review_content'] = amazon_df['review_content'].fillna("").str.lower()

# Keep useful columns
amazon_df = amazon_df[['product_name','category','actual_price','discounted_price','rating','review_content']]


## 3. Embedding

### -> load a sentiment model, sample reviews, truncate them, run analysis, and print the results with labels and scores

In [ ]:
from transformers import pipeline

# Load sentiment pipeline
sentiment = pipeline("sentiment-analysis")

# Sample reviews safely
sample_reviews = amazon_df['review_content'].dropna().sample(10).tolist()
sample_reviews = [str(review)[:512] for review in sample_reviews]  # truncate

# Run sentiment analysis
results = sentiment(sample_reviews)

# Display results
for review, res in zip(sample_reviews, results):
    print(f"Review: {review[:80]}... | Sentiment: {res['label']} | Score: {res['score']:.2f}")






### -> Handling errors

In [ ]:
# Ensure numeric prices
flipkart_df['discounted_price'] = pd.to_numeric(flipkart_df['discounted_price'], errors='coerce')
amazon_df['discounted_price'] = amazon_df['discounted_price'].astype(str).str.replace('₹','').str.replace(',','')
amazon_df['discounted_price'] = pd.to_numeric(amazon_df['discounted_price'], errors='coerce')

# Merge datasets
merged = flipkart_df.merge(amazon_df, on="product_name", how="inner")

# Calculate price gap
merged['price_gap'] = merged['discounted_price_x'] - merged['discounted_price_y']

print(merged[['product_name','discounted_price_x','discounted_price_y','price_gap']].head())




In [ ]:
print(amazon_df.columns.tolist())


### -> Plot the graph : "Demand vs Supply Trends by Category"

In [ ]:
import matplotlib.pyplot as plt

# Clean actual and discounted prices
amazon_df['actual_price'] = amazon_df['actual_price'].astype(str).str.replace('₹','').str.replace(',','')
amazon_df['actual_price'] = pd.to_numeric(amazon_df['actual_price'], errors='coerce')

amazon_df['discounted_price'] = pd.to_numeric(amazon_df['discounted_price'], errors='coerce')

# Supply proxy = discount percentage
amazon_df['discount_percentage'] = ((amazon_df['actual_price'] - amazon_df['discounted_price']) / amazon_df['actual_price']) * 100

# Demand proxy = rating
amazon_df['rating'] = pd.to_numeric(amazon_df['rating'], errors='coerce')

# Group by category
trend_df = amazon_df.groupby('category').agg({
    'rating':'mean',
        'discount_percentage':'mean'
        }).reset_index()

        # Plot demand vs supply
plt.figure(figsize=(10,6))
plt.scatter(trend_df['rating'], trend_df['discount_percentage'])
plt.xlabel("Average Demand (Rating)")
plt.ylabel("Average Supply Pressure (Discount %)")
plt.title("Demand vs Supply Trends by Category")
plt.show()




## 4. FAISS Index


In [ ]:
!pip install faiss-cpu


In [ ]:
import faiss


### -> finds the closest‑matching products by converting text into embeddings and ranking them with cosine similarity

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Force sentence-level embeddings (2D array)
flipkart_embeddings = model.encode(
    flipkart_df['description'].astype(str).tolist(),
        convert_to_numpy=True
        )

amazon_embeddings = model.encode(
            amazon_df['review_content'].astype(str).tolist(),
                convert_to_numpy=True
                )

# Encode query
query_vec = model.encode("your query text", convert_to_numpy=True)

# Compute similarity
similarities = cosine_similarity([query_vec], flipkart_embeddings)[0]

# Top 5 matches
top_indices = np.argsort(similarities)[-5:][::-1]
print(flipkart_df.iloc[top_indices][['product_name','description']])



### -> SentenceTransforme: finds the reviews most closely related to your query by comparing their embeddings with cosine similarity and ranking the top matches

In [ ]:
# Compute similarity between query and Amazon reviews
amazon_embeddings = model.encode(
    amazon_df['review_content'].astype(str).tolist(),
        convert_to_numpy=True
        )

amazon_similarities = cosine_similarity([query_vec], amazon_embeddings)[0]
top_amazon_indices = np.argsort(amazon_similarities)[-5:][::-1]

print(amazon_df.iloc[top_amazon_indices][['review_content']])


## 5. sentiment analysis pipeline


### -> evaluates the emotional tone of the most relevant reviews by combining semantic similarity with sentiment analysis

In [ ]:
from transformers import pipeline
sentiment = pipeline("sentiment-analysis")

matched_reviews = amazon_df.iloc[top_amazon_indices]['review_content'].tolist()
sentiments = sentiment(matched_reviews)

for review, s in zip(matched_reviews, sentiments):
    print(f"Review: {review[:80]}... | Sentiment: {s['label']} | Score: {s['score']:.2f}")


### -> visualizes the proportion of positive and negative sentiments in the most relevant reviews using a bar chart

In [ ]:
import matplotlib.pyplot as plt
labels = [s['label'] for s in sentiments]
plt.bar(['POSITIVE','NEGATIVE'], [labels.count('POSITIVE'), labels.count('NEGATIVE')])
plt.title("Sentiment of Top Amazon Matches")
plt.show()


### -> ensures accurate sentiment classification for lengthy reviews by chunking text and aggregating results with majority voting

In [ ]:
from transformers import pipeline

# Load sentiment pipeline
sentiment = pipeline("sentiment-analysis")

# Define chunking function
def chunk_text(text, max_len=512):
    return [text[i:i+max_len] for i in range(0, len(text), max_len)]

    # Step 1: Expand reviews into chunks
expanded_reviews = []
for r in matched_reviews:   # <-- make sure matched_reviews is defined earlier
    chunks = chunk_text(str(r), 512)
    expanded_reviews.append(chunks)

# Step 2: Flatten chunks for sentiment analysis
flat_chunks = [chunk for chunks in expanded_reviews for chunk in chunks]

# Step 3: Run sentiment analysis on all chunks
chunk_results = sentiment(flat_chunks)

# Step 4: Aggregate sentiment per review (majority vote)
review_sentiments = []
i = 0
for chunks in expanded_reviews:
    results = chunk_results[i:i+len(chunks)]
    labels = [res['label'] for res in results]
    majority_label = max(set(labels), key=labels.count)
    review_sentiments.append(majority_label)
    i += len(chunks)

                                # Now you can attach review_sentiments to your DataFrame


### -> collects reviews for a given product, filters for negative ones, tokenizes the text, and uses a frequency counter to return the most common complaint keywords

In [ ]:
from collections import Counter

def top_complaints(product_id, n=5):
    # Get reviews for the product
    reviews = amazon_df[amazon_df['product_id'] == product_id]['review_content'].dropna().tolist()

    if len(reviews) == 0:
       return f"No reviews found for product_id {product_id}"

    # Truncate to 512 chars
    reviews = [str(r)[:512] for r in reviews]

    # Run sentiment
    results = sentiment(reviews)

    # Collect negative reviews only
    negative_reviews = [r for r, s in zip(reviews, results) if s['label'] == 'NEGATIVE']

    if len(negative_reviews) == 0:
       return f"No negative reviews found for product_id {product_id}"

    # Tokenize and count complaint keywords
    words = " ".join(negative_reviews).lower().split()
    common = Counter(words).most_common(n)

    return common

    # Example usage
print(top_complaints(product_id="SKU123"))


In [ ]:
print(amazon_df['product_id'].unique()[:10])  # see valid IDs


## 6. Competitive Gap Analysis

### -> compares Flipkart and Amazon products by encoding descriptions into embeddings, matching them via FAISS similarity search, and then aggregates price gaps and top complaints across matched products

In [ ]:
import pandas as pd
import numpy as np
import faiss
from collections import Counter
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def competitive_gap_analysis(flipkart_df, amazon_df, top_n=5, sample_size=50):
    # --- Step 0: Ensure numeric prices ---
    flipkart_df['discounted_price'] = pd.to_numeric(flipkart_df['discounted_price'], errors='coerce')
    amazon_df['discounted_price'] = (
                amazon_df['discounted_price']
                    .astype(str)
                    .str.replace('₹','')
                    .str.replace(',','')
                         )
    amazon_df['discounted_price'] = pd.to_numeric(amazon_df['discounted_price'], errors='coerce')

    # --- Step 1: Sample to reduce runtime ---
    flipkart_sample = flipkart_df.sample(sample_size, random_state=42)
    amazon_sample = amazon_df.sample(sample_size*4, random_state=42)

    # --- Step 2: Build FAISS index for Amazon product names ---
    amazon_embeddings = model.encode(amazon_sample['product_name'].astype(str).tolist(),
               convert_to_numpy=True)
    dim = amazon_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(amazon_embeddings)

    # --- Step 3: Query Flipkart products ---
    flipkart_embeddings = model.encode(flipkart_sample['product_name'].astype(str).tolist(),
                      convert_to_numpy=True)
    D, I = index.search(flipkart_embeddings, k=1)  # top-1 match

    # --- Step 4: Build comparison DataFrame ---
    comp_df = pd.DataFrame({
             'Flipkart Product': flipkart_sample['product_name'].values,
             'Flipkart Price': flipkart_sample['discounted_price'].values,
             'Amazon Product': amazon_sample.iloc[I.flatten()]['product_name'].values,
             'Amazon Price': amazon_sample.iloc[I.flatten()]['discounted_price'].values,
             'Similarity Score': D.flatten()
                      })

    # Ensure Amazon Price is numeric
    comp_df['Amazon Price'] = pd.to_numeric(comp_df['Amazon Price'], errors='coerce')

    # --- Step 5: Calculate price gap ---
    comp_df['Price Gap'] = comp_df['Flipkart Price'] - comp_df['Amazon Price']

    # --- Step 6: Collect complaints from matched Amazon reviews ---
    complaints = []
    for idx in I.flatten():
        reviews = amazon_sample.iloc[idx]['review_content']
        words = str(reviews).lower().split()
        complaints.extend(words)

    top_complaints = Counter(complaints).most_common(top_n)

    return comp_df, top_complaints

 # Example usage

comp_df, complaints = competitive_gap_analysis(flipkart_df, amazon_df)
print(comp_df.head())
print("Top complaints across matched products:", complaints)


## 7. Competitive Gap Analysis with Citations

###-> compares Flipkart and Amazon products using embeddings and FAISS similarity, calculates price gaps, extracts top complaints, and attaches actual review snippets as citations to provide evidence for the analysis

In [ ]:
import pandas as pd
import numpy as np
import faiss
from collections import Counter
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def competitive_gap_analysis(flipkart_df, amazon_df, top_n=5, sample_size=50):
    # --- Step 0: Ensure numeric prices ---
    flipkart_df['discounted_price'] = pd.to_numeric(flipkart_df['discounted_price'], errors='coerce')
    amazon_df['discounted_price'] = (
              amazon_df['discounted_price']
              .astype(str)
              .str.replace('₹','')
              .str.replace(',','')
          )
    amazon_df['discounted_price'] = pd.to_numeric(amazon_df['discounted_price'], errors='coerce')

    # --- Step 1: Sample to reduce runtime ---
    flipkart_sample = flipkart_df.sample(sample_size, random_state=42)
    amazon_sample = amazon_df.sample(sample_size*4, random_state=42)

    # --- Step 2: Build FAISS index for Amazon product names ---
    amazon_embeddings = model.encode(amazon_sample['product_name'].astype(str).tolist(),
                     convert_to_numpy=True)
    dim = amazon_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(amazon_embeddings)

    # --- Step 3: Query Flipkart products ---
    flipkart_embeddings = model.encode(flipkart_sample['product_name'].astype(str).tolist(),
                    convert_to_numpy=True)
    D, I = index.search(flipkart_embeddings, k=1)  # top-1 match

    # --- Step 4: Build comparison DataFrame ---
    comp_df = pd.DataFrame({
           'Flipkart Product': flipkart_sample['product_name'].values,
           'Flipkart Price': flipkart_sample['discounted_price'].values,
           'Amazon Product': amazon_sample.iloc[I.flatten()]['product_name'].values,
           'Amazon Price': amazon_sample.iloc[I.flatten()]['discounted_price'].values,
           'Similarity Score': D.flatten()
                           })
    comp_df['Amazon Price'] = pd.to_numeric(comp_df['Amazon Price'], errors='coerce')
    comp_df['Price Gap'] = comp_df['Flipkart Price'] - comp_df['Amazon Price']

    # --- Step 5: Collect complaints + citations ---
    complaints = []
    citations = []
    for idx in I.flatten():
        reviews = amazon_sample.iloc[idx]['review_content']
        words = str(reviews).lower().split()
        complaints.extend(words)
        citations.append(reviews)  # store actual review text

    top_complaints = Counter(complaints).most_common(top_n)

    return comp_df, top_complaints, citations

# Example usage
comp_df, complaints, citations = competitive_gap_analysis(flipkart_df, amazon_df)
print(comp_df.head())
print("Top complaints across matched products:", complaints)
print("\nSample citations from reviews:")
for c in citations[:5]:
    print("-", c[:120], "...")


## 8. Structured Insight Reports


### -> Takeing the matched product data, top complaints, and citations, then formats them into a structured report with price analysis, complaint evidence, confidence metrics, and recommendations for a specific category

In [ ]:
def generate_insight_report(comp_df, top_complaints, citations, category="General"):
    report = f"""
      📊 Competitive Gap Report – Category: {category}

      💰 Price Analysis:
      - Flipkart Avg Price: ₹{comp_df['Flipkart Price'].mean():.2f}
      - Amazon Avg Price: ₹{comp_df['Amazon Price'].mean():.2f}
      - Avg Price Gap: ₹{comp_df['Price Gap'].mean():.2f}

      🗣️ Top Complaints (with citations):
      """
    for i, (word, count) in enumerate(top_complaints, 1):
        citation = citations[i-1][:120] if i-1 < len(citations) else "No citation available"
        report += f"\n{i}. {word} ({count} mentions)\n   Citation: \"{citation}...\""

    report += f"""

        ✅ Confidence:
         - Products analyzed: {len(comp_df)}
         - Reviews sampled: {len(citations)}
        - Similarity scores range: {comp_df['Similarity Score'].min():.2f} – {comp_df['Similarity Score'].max():.2f}
                              """
    return report

# Example usage
print(generate_insight_report(comp_df, complaints, citations, category="Headphones"))


### -> Generates a formatted report that summarizes average prices, price gaps, top complaints with citations, and confidence metrics for a chosen product category

In [ ]:
def generate_insight_report(comp_df, top_complaints, citations, category="General"):
    report = f"""
      📊 Competitive Gap Report – Category: {category}

      💰 Price Analysis:
      - Flipkart Avg Price: ₹{comp_df['Flipkart Price'].mean():.2f}
      - Amazon Avg Price: ₹{comp_df['Amazon Price'].mean():.2f}
      - Avg Price Gap: ₹{comp_df['Price Gap'].mean():.2f}

      🗣️ Top Complaints (with citations):
      """
    for i, (word, count) in enumerate(top_complaints, 1):
        citation = citations[i-1][:120] if i-1 < len(citations) else "No citation available"
        report += f"\n{i}. {word} ({count} mentions)\n   Citation: \"{citation}...\""

    report += f"""

       ✅ Confidence:
       - Products analyzed: {len(comp_df)}
       - Reviews sampled: {len(citations)}
       - Similarity scores range: {comp_df['Similarity Score'].min():.2f} – {comp_df['Similarity Score'].max():.2f}
                              """
    return report

# Example usage
print(generate_insight_report(comp_df, complaints, citations, category="Headphones"))


### -> Builds a formatted competitive gap report that summarizes average prices, price gaps, top complaints with citations, and confidence metrics for a chosen category, making the analysis business‑ready and easy to read.

In [ ]:
def generate_insight_report(comp_df, top_complaints, citations, category="General"):
    # Build report header
    report = f"""
    📊 Competitive Gap Report – Category: {category}

    💰 Price Analysis:
     - Flipkart Avg Price: ₹{comp_df['Flipkart Price'].mean():.2f}
     - Amazon Avg Price: ₹{comp_df['Amazon Price'].mean():.2f}
     - Avg Price Gap: ₹{comp_df['Price Gap'].mean():.2f}
          """

     # Add complaints with citations
    report += "\n🗣️ Top Complaints (with citations):"
    for i, (word, count) in enumerate(top_complaints, 1):
        citation = citations[i-1][:120] if i-1 < len(citations) else "No citation available"
        report += f"\n{i}. {word} ({count} mentions)\n   Citation: \"{citation}...\""

    # Add confidence metrics
    report += f"""

    ✅ Confidence:
   - Products analyzed: {len(comp_df)}
   - Reviews sampled: {len(citations)}
   - Similarity scores range: {comp_df['Similarity Score'].min():.2f} – {comp_df['Similarity Score'].max():.2f}
               """

     # Add recommendations
    avg_gap = comp_df['Price Gap'].mean()
    if avg_gap > 0:
       recommendation = "Flipkart products are priced higher on average. Consider reducing prices or adding premium features."
    elif avg_gap < 0:
        recommendation = "Flipkart products are priced lower on average. Highlight value-for-money in marketing."
    else:
        recommendation = "Prices are aligned. Focus on differentiating features and customer experience."

    report += f"\n📌 Recommendation:\n- {recommendation}\n"

    return report

 # Example usage
print(generate_insight_report(comp_df, complaints, citations, category="Headphones"))


## 8. a. Citations

### -> generates a competitive gap report that summarizes price analysis, top complaints, and citations, while also adapting recommendations based on the chosen business goal

In [ ]:
def generate_insight_report(comp_df, top_complaints, citations, category="General", goal="growth"):
    report = f"""
    📊 Competitive Gap Report – Category: {category}

    💰 Price Analysis:
    - Flipkart Avg Price: ₹{comp_df['Flipkart Price'].mean():.2f}
    - Amazon Avg Price: ₹{comp_df['Amazon Price'].mean():.2f}
    - Avg Price Gap: ₹{comp_df['Price Gap'].mean():.2f}

    🗣️ Top Complaints (with citations):
     """
    for i, (word, count) in enumerate(top_complaints, 1):
        citation = citations[i-1][:120] if i-1 < len(citations) else "No citation available"
        report += f"\n{i}. {word} ({count} mentions)\n   Citation: \"{citation}...\""

    # Confidence metrics
    report += f"""

      ✅ Confidence:
     - Products analyzed: {len(comp_df)}
    - Reviews sampled: {len(citations)}
    - Similarity scores range: {comp_df['Similarity Score'].min():.2f} – {comp_df['Similarity Score'].max():.2f}
             """

   # Goal‑aware recommendations
    if goal == "growth":
       recommendation = "Focus on reducing price gaps and improving customer satisfaction to drive volume."
    elif goal == "margins":
       recommendation = "Maintain higher price points but address top complaints to justify premium positioning."
    elif goal == "retention":
       recommendation = "Prioritize fixing recurring complaints to improve customer loyalty."
    else:
       recommendation = "Balance pricing and features based on category trends."

    report += f"\n📌 Recommendation (Goal: {goal}):\n- {recommendation}\n"

    return report

# Example usage
print(generate_insight_report(comp_df, complaints, citations, category="Headphones", goal="margins"))


## 8. b. Domain‑Aware Memory


### -> It applies user‑defined filters (like price thresholds) to the competitive gap dataframe and regenerates the structured insight report with updated category and goal focus.

In [ ]:
def interactive_analysis(comp_df, complaints, citations, category="General", goal="growth", filters=None):
      # Apply filters dynamically
     df = comp_df.copy()
     if filters:
        for col, condition in filters.items():
        # Use backticks around column names with spaces
             df = df.query(f"`{col}`{condition}")

        # Generate report with updated goal
     return generate_insight_report(df, complaints, citations, category=category, goal=goal)

# Example usage:
# Switch to margins focus
print(interactive_analysis(comp_df, complaints, citations, category="Headphones", goal="margins"))

# Focus only on products above ₹5000
print(interactive_analysis(comp_df, complaints, citations, category="Smartphones", filters={"Flipkart Price": ">5000"}))
